<a href="https://colab.research.google.com/github/mihnea-popescu/yolov1-cupy/blob/linear_class/notebooks/linear.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/mihnea-popescu/yolov1-cupy.git
%cd yolov1-cupy
# !git checkout linear_class

Cloning into 'yolov1-cupy'...
remote: Enumerating objects: 35, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 35 (delta 9), reused 30 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (35/35), 15.86 KiB | 7.93 MiB/s, done.
Resolving deltas: 100% (9/9), done.
/content/yolov1-cupy
Branch 'linear_class' set up to track remote branch 'linear_class' from 'origin'.
Switched to a new branch 'linear_class'


In [2]:
!pip install cupy-cuda12x
import cupy as cp
from linear import Linear

Test 1: Output shape is correct

In [4]:
layer = Linear(in_features=8, out_features=4)
x = cp.random.randn(16, 8)
out = layer(x)
assert out.shape == (16, 4), f"Expected (16, 4), got {out.shape}"
print("Test 1 passed: output shape correct")

Test 1 passed: output shape correct


Test 2: Forward pass values are correct

In [5]:
layer = Linear(in_features=4, out_features=3, bias=True)

# Manually set known weights and bias so we can verify by hand
layer.W = cp.array([[1, 0, 0, 0],
                    [0, 1, 0, 0],
                    [0, 0, 1, 0]], dtype=cp.float64)
layer.b = cp.zeros(3)

x = cp.array([[1, 2, 3, 4],
              [5, 6, 7, 8]], dtype=cp.float64)

out = layer(x)
expected = cp.array([[1, 2, 3],
                     [5, 6, 7]], dtype=cp.float64)

cp.testing.assert_allclose(out, expected, rtol=1e-5)
print("Test 2 passed: forward pass values correct")

Test 2 passed: forward pass values correct


Test 3: Backward pass gradient shape

In [6]:
layer = Linear(4, 3)
x = cp.random.randn(5, 4)
out = layer(x)

d_out = cp.ones_like(out)
d_input = layer.backward(d_out)

assert d_input.shape == x.shape,        f"d_input shape mismatch: {d_input.shape}"
assert layer.dW.shape == layer.W.shape, f"dW shape mismatch: {layer.dW.shape}"
assert layer.db.shape == layer.b.shape, f"db shape mismatch: {layer.db.shape}"
print("Test 3 passed: backward gradients have correct shapes")

Test 3 passed: backward gradients have correct shapes


Test 4: Backward pass gradient values are correct

In [7]:
layer = Linear(in_features=2, out_features=2, bias=True)

layer.W = cp.array([[1, 0],
                    [0, 1]], dtype=cp.float64)
layer.b = cp.zeros(2)

x = cp.array([[2, 3]], dtype=cp.float64)
out = layer(x)

d_out = cp.array([[1, 1]], dtype=cp.float64)
d_input = layer.backward(d_out)

# With identity W, d_input should equal d_out @ I = d_out
cp.testing.assert_allclose(d_input, d_out, rtol=1e-5)
print("Test 4 passed: backward gradient values correct")

Test 4 passed: backward gradient values correct


Test 5: No-bias mode works

In [8]:
layer = Linear(4, 3, bias=False)
x = cp.random.randn(5, 4)
out = layer(x)
assert out.shape == (5, 3)
assert layer.b is None
print("Test 5 passed: bias=False works correctly")

Test 5 passed: bias=False works correctly
